In [2]:
import numpy as np
from scipy.linalg import cho_factor, cho_solve, eigvalsh, solve
from scipy.special import roots_hermite
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from time import time

np.random.seed(42)

## 1. Model Setup

In [3]:
d = 10       # assets
T = 200      # training periods
h_fore = 20  # forecast horizon

# --- RBF temporal kernel ---
def rbf_kernel(t1, t2, length_scale=20.0, variance=1.0):
    t1 = np.asarray(t1).reshape(-1, 1)
    t2 = np.asarray(t2).reshape(1, -1)
    return variance * np.exp(-0.5 * (t1 - t2)**2 / length_scale**2)

times_train = np.arange(1, T + 1, dtype=float)
times_fore  = np.arange(T + 1, T + 1 + h_fore, dtype=float)

K_tt = rbf_kernel(times_train, times_train) + 1e-5 * np.eye(T)
K_ts = rbf_kernel(times_train, times_fore)
K_ss = rbf_kernel(times_fore, times_fore)  + 1e-5 * np.eye(h_fore)
K_tt_cho = cho_factor(K_tt)
K_t_inv  = cho_solve(K_tt_cho, np.eye(T))

# --- Asset graph ---
def make_asset_graph(d, p_edge=0.4, seed=123):
    rng = np.random.RandomState(seed)
    W = np.zeros((d, d))
    perm = rng.permutation(d)
    for k in range(d - 1):
        w = rng.uniform(0.3, 1.0)
        W[perm[k], perm[k+1]] = w
        W[perm[k+1], perm[k]] = w
    for i in range(d):
        for j in range(i+1, d):
            if W[i,j] == 0 and rng.rand() < p_edge:
                w = rng.uniform(0.1, 1.0)
                W[i,j] = w; W[j,i] = w
    D = np.diag(W.sum(axis=1))
    return D - W, W

L_graph, W_graph = make_asset_graph(d)
alpha_g, beta_g = 1.5, 1.0
Q_g     = alpha_g * L_graph + beta_g * np.eye(d)
Q_g_inv = np.linalg.inv(Q_g)

# Full prior precision Q = K_t^{-1} ⊗ Q_g  (N x N, N = dT)
N = d * T
Q_full = np.kron(K_t_inv, Q_g)

# ============================================================
# 2. SIMULATE DATA
# ============================================================
print("=" * 70)
print("1. SIMULATING DATA")
print("=" * 70)

L_Qg = np.linalg.cholesky(Q_g_inv)
L_Kt = np.linalg.cholesky(K_tt)

H_true = L_Qg @ np.random.randn(d, T) @ L_Kt.T   # d x T
S_true = H_true.copy()
V_true = np.exp(S_true)

mu = np.zeros(d)
R  = mu[:, None] + np.sqrt(V_true) * np.random.randn(d, T)

# Future truth for evaluation
A_time = cho_solve(K_tt_cho, K_ts)          # T x h
K_cond = K_ss - K_ts.T @ A_time             # h x h
L_cond = np.linalg.cholesky(K_cond + 1e-7 * np.eye(h_fore))
H_future_true = H_true @ A_time + L_Qg @ np.random.randn(d, h_fore) @ L_cond.T
S_future_true = H_future_true.copy()
V_future_true = np.exp(S_future_true)

print(f"  d={d}, T={T}, h_fore={h_fore}")
print(f"  Log-vol range:  [{S_true.min():.2f}, {S_true.max():.2f}]")
print(f"  Vol range:      [{V_true.min():.4f}, {V_true.max():.4f}]")


1. SIMULATING DATA
  d=10, T=200, h_fore=20
  Log-vol range:  [-1.45, 1.64]
  Vol range:      [0.2338, 5.1513]


## 2. Simulate Data

## 3. Likelihood Functions

## 4. MAP Estimation (ICM / Newton)

## 5. Laplace Approximation

## 6. Expectation Propagation

## 7. Forecasting

## 8. Evaluation Metrics 

## 9. Plots